<div style="background: linear-gradient(135deg, #034694 0%, #1E8449 50%, #D4AC0D 100%); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
    <h1 style="color: #FFF; text-shadow: 1px 1px 3px rgba(0,0,0,0.5);">💬 | Step 2: Customize Tone With SFT </h1>
        <p style="font-size: 16px; line-height: 1.6;">
            We used few shot examples to prompt-engineer a better tone. We used RAG to ground responses in our data. But this keeps growing our prompt lengths (increasing token costs and reduce effective context window available for output). How can we improve the tone and style of our bot with _more examples_ and shorter prompt length?
        </p>
</div>

<div style="display: flex; align-items: center; justify-content: left; padding: 5px; height: 40px; background: linear-gradient(90deg, #333333 0%, #777777 50%, #BBBBBB 100%); border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 1.5em; font-weight: bold; color: #fff;">
    Step 1 : Understand Zava Scenarios
</div>
<br/>
<div style="display: flex; align-items: center; justify-content: left; padding: 5px; height: 40px; background: linear-gradient(90deg, #7873f5 0%, #ff6ec4 100%); border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 1.5em; font-weight: bold; color: #fff;">
    Step 3: Be More Cost-Effective With Distillation
</div>
<br/>
<div style="display: flex; align-items: center; justify-content: left; padding: 5px; height: 40px; background: linear-gradient(90deg, #7873f5 0%, #ff6ec4 100%); border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 1.5em; font-weight: bold; color: #fff;">
    Step 4: Be More Precise with RAFT
</div>

---
### 1. Check Environment Variables

In [1]:
import os

openai_key = os.getenv("AZURE_OPENAI_API_KEY")
openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
model_name = "gpt-4o"
api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2025-02-01-preview")

if not openai_key or not openai_endpoint:
    print("Error: Missing AZURE_OPENAI_KEY or AZURE_OPENAI_ENDPOINT environment variable.")

print("Using Model:", model_name)
print("Using API Version:", api_version)

Using Model: gpt-4o
Using API Version: 2025-02-01-preview


---
### 2. Validate Training Dataset

In [4]:
# Identify Training and Validation datafiles

training_file = "01-basic_sft_training.jsonl" 
validation_file = "01-basic_sft_validation.jsonl"

In [5]:
# Run preliminary checks

import json

# Load the training set
with open(training_file, 'r', encoding='utf-8') as f:
    training_dataset = [json.loads(line) for line in f]

# Training dataset stats
print("Number of examples in training set:", len(training_dataset))
print("First example in training set:")
for message in training_dataset[0]["messages"]:
    print(message)

# Load the validation set
with open(validation_file, 'r', encoding='utf-8') as f:
    validation_dataset = [json.loads(line) for line in f]

# Validation dataset stats
print("\nNumber of examples in validation set:", len(validation_dataset))
print("First example in validation set:")
for message in validation_dataset[0]["messages"]:
    print(message)

Number of examples in training set: 40
First example in training set:
{'role': 'system', 'content': 'You are Cora, a polite, factual and helpful assistant for Zava, a DIY hardware store.'}
{'role': 'user', 'content': 'Is water-based polyurethane better than oil-based?'}
{'role': 'assistant', 'content': '🤔 Both have benefits! Water-based at $43 dries faster, oil-based at $34 gives amber tone. Which matters most?'}

Number of examples in validation set: 10
First example in validation set:
{'role': 'system', 'content': 'You are Cora, a polite, factual and helpful assistant for Zava, a DIY hardware store.'}
{'role': 'user', 'content': "What's good for painting wrought iron?"}
{'role': 'assistant', 'content': '🔧 Excellent combo! Metal Primer at $36 first, then Gloss Spray Paint at $8. Rust removal first?'}


---
### 3. Assess Token Counts For Data

In [6]:
# Validate token counts

import json
import tiktoken
import numpy as np

encoding = tiktoken.get_encoding("o200k_base") # default encoding for gpt-4o models. This requires the latest version of tiktoken to be installed.

def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1):
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3
    return num_tokens

def num_assistant_tokens_from_messages(messages):
    num_tokens = 0
    for message in messages:
        if message["role"] == "assistant":
            num_tokens += len(encoding.encode(message["content"]))
    return num_tokens

def print_distribution(values, name):
    print(f"\n#### Distribution of {name}:")
    print(f"min / max: {min(values)}, {max(values)}")
    print(f"mean / median: {np.mean(values)}, {np.median(values)}")
    print(f"p5 / p95: {np.quantile(values, 0.1)}, {np.quantile(values, 0.9)}")

files = [training_file, validation_file]

for file in files:
    print(f"Processing file: {file}")
    with open(file, 'r', encoding='utf-8') as f:
        dataset = [json.loads(line) for line in f]

    total_tokens = []
    assistant_tokens = []

    for ex in dataset:
        messages = ex.get("messages", {})
        total_tokens.append(num_tokens_from_messages(messages))
        assistant_tokens.append(num_assistant_tokens_from_messages(messages))

    print_distribution(total_tokens, "total tokens")
    print_distribution(assistant_tokens, "assistant tokens")
    print('*' * 50)

Processing file: 01-basic_sft_training.jsonl

#### Distribution of total tokens:
min / max: 62, 78
mean / median: 68.15, 68.0
p5 / p95: 65.0, 71.1

#### Distribution of assistant tokens:
min / max: 19, 30
mean / median: 23.525, 23.0
p5 / p95: 20.9, 26.1
**************************************************
Processing file: 01-basic_sft_validation.jsonl

#### Distribution of total tokens:
min / max: 63, 76
mean / median: 68.3, 67.5
p5 / p95: 65.7, 70.6

#### Distribution of assistant tokens:
min / max: 20, 28
mean / median: 23.3, 23.0
p5 / p95: 21.8, 25.299999999999997
**************************************************


---
### 4. Upload Fine-Tuning Data To Cloud

In [7]:
# Create Azure OpenAI Client

import os
from openai import AzureOpenAI

client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
  api_key = os.getenv("AZURE_OPENAI_API_KEY"),
  api_version = os.getenv("AZURE_OPENAI_API_VERSION")
)

In [8]:
# Upload the training and validation dataset files to Azure OpenAI with the SDK.
# Note: You can visit the Microsoft Foundry Portal - and look under your Azure AI Project's 'Data Files' tab to see the uploaded files.

training_response = client.files.create(
    file = open(training_file, "rb"), purpose="fine-tune"
)
training_file_id = training_response.id

validation_response = client.files.create(
    file = open(validation_file, "rb"), purpose="fine-tune"
)
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)

Training file ID: file-72d726c78784421c90ba05334f359399
Validation file ID: file-866579aa97df4ea68b53ead6265228c6


---
### 5. Submit The Fine-Tuning Job

In [10]:
# Submit fine-tuning training job
# Note that the model you specify here must be one that can be fine-tuned.
# See: https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/fine-tuning
response = client.fine_tuning.jobs.create(
    training_file=training_file_id,
    validation_file=validation_file_id,
    model="gpt-4o", # Enter base model name. Note that in Azure OpenAI the model name contains dashes and cannot contain dot/period characters.
    seed = 105  # seed parameter controls reproducibility of the fine-tuning job. If no seed is specified one will be generated automatically.
)

job_id = response.id

# You can use the job ID to monitor the status of the fine-tuning job.
# The fine-tuning job will take some time to start and complete.
print("Job ID:", response.id)
print("Status:", response.status)
print(response.model_dump_json(indent=2))

Job ID: ftjob-00251b79b96641d888eccf524f8e3498
Status: pending
{
  "id": "ftjob-00251b79b96641d888eccf524f8e3498",
  "created_at": 1767603687,
  "error": null,
  "fine_tuned_model": null,
  "finished_at": null,
  "hyperparameters": {
    "batch_size": -1,
    "learning_rate_multiplier": 1.0,
    "n_epochs": -1
  },
  "model": "gpt-4o-2024-08-06",
  "object": "fine_tuning.job",
  "organization_id": null,
  "result_files": null,
  "seed": 105,
  "status": "pending",
  "trained_tokens": null,
  "training_file": "file-72d726c78784421c90ba05334f359399",
  "validation_file": "file-866579aa97df4ea68b53ead6265228c6",
  "estimated_finish": 1767604588,
  "integrations": null,
  "metadata": null,
  "method": null
}


---

### 6. Track Fine-Tuning Job Status

In [11]:
# Track training status

from IPython.display import clear_output
import time

start_time = time.time()

# Get the status of our fine-tuning job.
response = client.fine_tuning.jobs.retrieve(job_id)

status = response.status

# If the job isn't done yet, poll it every 10 seconds.
while status not in ["succeeded", "failed"]:
    time.sleep(10)

    response = client.fine_tuning.jobs.retrieve(job_id)
    print(response.model_dump_json(indent=2))
    print("Elapsed time: {} minutes {} seconds".format(int((time.time() - start_time) // 60), int((time.time() - start_time) % 60)))
    status = response.status
    print(f'Status: {status}')
    clear_output(wait=True)

print(f'Fine-tuning job {job_id} finished with status: {status}')

# List all fine-tuning jobs for this resource.
print('Checking other fine-tune jobs for this resource.')
response = client.fine_tuning.jobs.list()
print(f'Found {len(response.data)} fine-tune jobs.')

Fine-tuning job ftjob-00251b79b96641d888eccf524f8e3498 finished with status: succeeded
Checking other fine-tune jobs for this resource.
Found 15 fine-tune jobs.


---

### 7. List Fine-Tuning Events

In [12]:
response = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)
print(response.model_dump_json(indent=2))

{
  "data": [
    {
      "id": "ftevent-f608932e564d45ba85f7c492adacc21a",
      "created_at": 1767607538,
      "level": "info",
      "message": "Training tokens billed: 8000",
      "object": "fine_tuning.job.event",
      "data": null,
      "type": "message"
    },
    {
      "id": "ftevent-a00b1bb307c447a28f4af2f9f34f589b",
      "created_at": 1767607537,
      "level": "info",
      "message": "Model Evaluation Passed.",
      "object": "fine_tuning.job.event",
      "data": null,
      "type": "message"
    },
    {
      "id": "ftevent-f8b47667cd834a4990f3618f02a89345",
      "created_at": 1767607537,
      "level": "info",
      "message": "Completed results file: file-8bc75b049ce446dbba12abd7d6d82a1f",
      "object": "fine_tuning.job.event",
      "data": null,
      "type": "message"
    },
    {
      "id": "ftevent-4d208e527472454c90d2d2cb11a23bf5",
      "created_at": 1767607459,
      "level": "info",
      "message": "Job succeeded.",
      "object": "fine_tuning.jo

---

### 8. List Fine-Tuning Checkpoints

In [13]:
response = client.fine_tuning.jobs.checkpoints.list(job_id)
print(response.model_dump_json(indent=2))

{
  "data": [
    {
      "id": "ftchkpt-e672663597c843b28092b7bd87cc2f44",
      "created_at": 1767606304,
      "fine_tuned_model_checkpoint": "gpt-4o-2024-08-06.ft-00251b79b96641d888eccf524f8e3498",
      "fine_tuning_job_id": "ftjob-00251b79b96641d888eccf524f8e3498",
      "metrics": {
        "full_valid_loss": 1.3638394698795122,
        "full_valid_mean_token_accuracy": 0.6521739130434783,
        "step": 120.0,
        "train_loss": 1.3354382514953613,
        "train_mean_token_accuracy": 0.6666666865348816,
        "valid_loss": 1.287997283935547,
        "valid_mean_token_accuracy": 0.68
      },
      "object": "fine_tuning.job.checkpoint",
      "step_number": 120
    },
    {
      "id": "ftchkpt-29c48050e51e4c9da1dfa88d76bf82bd",
      "created_at": 1767606113,
      "fine_tuned_model_checkpoint": "gpt-4o-2024-08-06.ft-00251b79b96641d888eccf524f8e3498:ckpt-step-80",
      "fine_tuning_job_id": "ftjob-00251b79b96641d888eccf524f8e3498",
      "metrics": {
        "full_vali

---

### 9. Retrieve Fine-Tuned Model Name

In [14]:
# Retrieve fine_tuned_model name

response = client.fine_tuning.jobs.retrieve(job_id)

print(response.model_dump_json(indent=2))
fine_tuned_model = response.fine_tuned_model

{
  "id": "ftjob-00251b79b96641d888eccf524f8e3498",
  "created_at": 1767603687,
  "error": null,
  "fine_tuned_model": "gpt-4o-2024-08-06.ft-00251b79b96641d888eccf524f8e3498",
  "finished_at": 1767607538,
  "hyperparameters": {
    "batch_size": 1,
    "learning_rate_multiplier": 1.0,
    "n_epochs": 3
  },
  "model": "gpt-4o-2024-08-06",
  "object": "fine_tuning.job",
  "organization_id": null,
  "result_files": [
    "file-8bc75b049ce446dbba12abd7d6d82a1f"
  ],
  "seed": 105,
  "status": "succeeded",
  "trained_tokens": 10440,
  "training_file": "file-72d726c78784421c90ba05334f359399",
  "validation_file": "file-866579aa97df4ea68b53ead6265228c6",
  "estimated_finish": 1767605042,
  "integrations": null,
  "metadata": null,
  "method": null
}


---

### 10. Deploy Fine-Tuned Model For Testing

For now - we will deploy our model to the Microsoft Foundry Portal - using the **developer tier** which allows us to test our fine-tuned model for the cost of just inferencing. Once we deploy it, we can try it out

### Configure deployment settings

In [15]:
import time

# Generate unique deployment name
timestamp = int(time.time())
DEPLOYMENT_NAME = f"02-zava-finetuned-{timestamp}"

# Configure deployment settings
DEPLOYMENT = {
    "properties": {
        "model": { 
            "format": "OpenAI", 
            "name": fine_tuned_model, 
            "version": "1" 
        },
    },
    "sku": { 
        "capacity": 8,  # Adjust based on your needs (e.g., 250 for DeveloperTier)
        "name": "GlobalStandard"  # Options: "DeveloperTier", "Standard", "GlobalStandard"
    },
}

print(f"📋 Deployment Configuration:")
print(f"   Name: {DEPLOYMENT_NAME}")
print(f"   Model: {fine_tuned_model}")
print(f"   SKU: {DEPLOYMENT['sku']['name']}")
print(f"   Capacity: {DEPLOYMENT['sku']['capacity']}")

📋 Deployment Configuration:
   Name: 02-zava-finetuned-1767607550
   Model: gpt-4o-2024-08-06.ft-00251b79b96641d888eccf524f8e3498
   SKU: GlobalStandard
   Capacity: 8


In [16]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

# Create management client for Azure Cognitive Services
cogsvc_client = CognitiveServicesManagementClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.environ.get("AZURE_SUBSCRIPTION_ID")
)

print("✅ Azure Management client created successfully!")

✅ Azure Management client created successfully!


### Deploy Fine Tuned model

In [17]:
# Submit deployment request
print(f"🚀 Starting deployment of {DEPLOYMENT_NAME}...\n")

deployment = cogsvc_client.deployments.begin_create_or_update(
    resource_group_name=os.environ.get("AZURE_RESOURCE_GROUP"),
    account_name=os.environ.get("AZURE_AI_FOUNDRY_NAME"),
    deployment_name=DEPLOYMENT_NAME,
    deployment=DEPLOYMENT,
)

print(f"✅ Deployment request submitted!")
print(f"\n⏳ Deployment is now provisioning...")
print(f"   This typically takes 3-5 minutes for small models")

🚀 Starting deployment of 02-zava-finetuned-1767607550...



✅ Deployment request submitted!

⏳ Deployment is now provisioning...
   This typically takes 3-5 minutes for small models


In [18]:
from IPython.display import clear_output
import time

start_time = time.time()
status = deployment.status()

while status not in ["Succeeded", "Failed"]:
    deployment.wait(5)
    status = deployment.status()
    elapsed_min = int((time.time() - start_time) // 60)
    elapsed_sec = int((time.time() - start_time) % 60)
    
    clear_output(wait=True)
    print(f"🛳️  Provisioning {DEPLOYMENT_NAME}")
    print(f"📊 Status: {status}")
    print(f"⏱️  Elapsed time: {elapsed_min} minutes {elapsed_sec} seconds")

# Final status
elapsed_min = int((time.time() - start_time) // 60)
elapsed_sec = int((time.time() - start_time) % 60)

if status == "Succeeded":
    print(f"\n🎉 Deployment completed successfully!")
    print(f"⏱️  Total time: {elapsed_min} minutes {elapsed_sec} seconds")
    print(f"\n📝 Deployment Details:")
    print(f"   Name: {DEPLOYMENT_NAME}")
    print(f"   Model: {fine_tuned_model}")
else:
    print(f"\n❌ Deployment failed with status: {status}")

🛳️  Provisioning 02-zava-finetuned-1767607550
📊 Status: Succeeded
⏱️  Elapsed time: 10 minutes 32 seconds

🎉 Deployment completed successfully!
⏱️  Total time: 10 minutes 32 seconds

📝 Deployment Details:
   Name: 02-zava-finetuned-1767607550
   Model: gpt-4o-2024-08-06.ft-00251b79b96641d888eccf524f8e3498


### Test the deployed model

In [19]:
# Test the deployed model with multiple sample prompts
test_prompts = [
    "What kind of paint should I buy for my outdoor deck?Can I use extension poles with your roller frames?",
    "I'm painting over rust - what spray paint should I use?"
]

for i, test_prompt in enumerate(test_prompts, 1):
    print(f"Test {i}/{len(test_prompts)}: Testing deployed model with prompt:")
    print(f"   '{test_prompt}'\n")
    
    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,  # Use the deployment name
        messages=[
            {"role": "system", "content": "You are Cora, a polite, factual and helpful assistant for Zava, a DIY hardware store."},
            {"role": "user", "content": test_prompt}
        ],
        max_tokens=150
    )
    
    print(f"Response from {DEPLOYMENT_NAME}:")
    print(response.choices[0].message.content)
    print("\n" + "="*80 + "\n")

Test 1/2: Testing deployed model with prompt:
   'What kind of paint should I buy for my outdoor deck?Can I use extension poles with your roller frames?'



Response from 02-zava-finetuned-1767607550:
🎨Outdoor wood paint at $40! Deck Stain with UV protection also available? Extension pole at $18, supports all frames.


Test 2/2: Testing deployed model with prompt:
   'I'm painting over rust - what spray paint should I use?'

Response from 02-zava-finetuned-1767607550:
🔩 Perfect! Rust-Blocking Spray Paint at $14, in various colors, includes a primer for direct rust application.👍




**Insights**

In both the examples above we can note that the response now accurately follows our Zava guidelines for "polite, factual and helpful"
- Every response starts with an emoji
- The first sentence is always an acknowledgement of the user ("polite")
- The next sentence is always an informative segment ("factual")
- The final senteance is always an offer to follow up ("helpful")

And note that we have the succinct responses we were looking for _without adding few-shot examples_, making the prompts shorter and thus saving both token costs and processing latency.

---
### Teardown

Once you are done with this lab, don't forget to tear down the infrastructure. The developer tier model will be torn down automatically (after 24 hours?) but it is better to proactively delete the resource group and release all model quota.


<div style="display: flex; align-items: center; justify-content: left; padding: 5px; height: 40px; background: linear-gradient(90deg, #7873f5 0%, #ff6ec4 100%); border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 1.5em; font-weight: bold; color: #fff;">
    Next: Be More Cost-Effective With Distillation
</div>